<a href="https://colab.research.google.com/github/rj05-ux/customer-segmentation-rfm/blob/main/notebooks/03_RFM_analysis_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 RFM Feature Engineering

## Objective

To transform transaction-level data into customer-level features using RFM analysis.

## RFM Definition

* Recency: Days since last purchase
* Frequency: Number of purchases
* Monetary: Total spending

## Why RFM?

Raw transaction data does not directly represent customer behavior. RFM summarizes customer activity in a meaningful way.

## Steps

1. Group data by CustomerID
2. Calculate Recency, Frequency, Monetary
3. Apply log transformation
4. Scale features

## Output

A customer-level dataset ready for clustering.


**STEP 1 — Load Clean Data**

In [1]:
import pandas as pd

df = pd.read_csv('/content/Online Retail Data Set.csv', encoding='ISO-8859-1')

df = df.dropna(subset=['CustomerID'])
df = df[df['Quantity'] > 0]
df = df[df['UnitPrice'] > 0]

df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], dayfirst=True)
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']

**STEP 2 — Define Snapshot Date**

In [4]:
import datetime as dt

snapshot_date = df['InvoiceDate'].max() + dt.timedelta(days=1)
print(snapshot_date)

2011-12-10 12:50:00


**STEP 3 — Create RFM Table**

In [5]:
rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,
    'InvoiceNo': 'count',
    'TotalPrice': 'sum'
})

rfm.columns = ['Recency', 'Frequency', 'Monetary']

rfm.head()

,Recency,Frequency,Monetary
CustomerID,,,
12346.0,326,1,77183.60
12347.0,2,182,4310.00
12348.0,75,31,1797.24
12349.0,19,73,1757.55
12350.0,310,17,334.40


**STEP 4 — CHECK DISTRIBUTION**

In [6]:
rfm.describe()

,Recency,Frequency,Monetary
count,4338.000000,4338.000000,4338.000000
mean,92.536422,91.720609,2054.266460
std,100.014169,228.785094,8989.230441
min,1.000000,1.000000,3.750000
25%,18.000000,17.000000,307.415000
50%,51.000000,41.000000,674.485000
75%,142.000000,100.000000,1661.740000
max,374.000000,7847.000000,280206.020000


**RFM Observations**



*   Monetary and Frequency are highly skewed with extreme outliers

*   A small number of customers contribute very high revenue
*   Recency varies widely, indicating both active and inactive customers


*   Dataset contains both high-value and low-value customers


**STEP 5 — LOG TRANSFORMATION**

In [7]:
import numpy as np

rfm_log = np.log1p(rfm)

**STEP 6 — SCALING**

In [8]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_log)